# Philippines Model

from Transition Zero scenario builder (3 node model with mix of planning and dispatch data)

In [18]:
import pypsa
import pandas as pd
import numpy as np

from pypsa.costs import annuity

In [19]:
transmission = pd.read_csv("../data/Installed Capacity.csv").query("year == 2025 and ~interconnector.isna()").dropna(how='all', axis=1)
transmission[["bus0", "bus1"]] = pd.DataFrame(transmission.interconnector_name.str.split(" ~ ").tolist(), index=transmission.index)
transmission.set_index("interconnector_name", inplace=True)
transmission = transmission.loc[["Luzon ~ Visayas", "Visayas ~ Mindanao"]]

In [20]:
annual_demand = pd.read_csv("../data/Annual Demand.csv").query("year == 2025").set_index("node_name")["default_data"]
demand_profile = pd.read_csv("../data/Demand Profile.csv")
demand_profile = demand_profile.set_index(["year_part", "day_part", "node_name"])["default_data"].unstack("node_name", sort=False)
demand_profile.index = pd.date_range("2025-01-01", "2026-01-01", inclusive="left", freq="h")
p_set = demand_profile * annual_demand
p_set.index = p_set.index.tz_localize("UTC").tz_convert("Asia/Manila").tz_localize(None)
p_set = p_set.resample("3h").mean()

In [21]:
p_max_pu = pd.read_csv("../data/Renewable Profile.csv")
p_max_pu = p_max_pu.set_index(["year_part", "day_part", "node_name", "technology"])["default_data"].unstack(["node_name", "technology"], sort=False)
p_max_pu.index = pd.date_range("2025-01-01", "2026-01-01", inclusive="left", freq="h")
p_max_pu = p_max_pu.loc[:, ~p_max_pu.eq(1).all(axis=0)]
p_max_pu.index = p_max_pu.index.tz_localize("UTC").tz_convert("Asia/Manila").tz_localize(None)
p_max_pu.columns = p_max_pu.columns.get_level_values("node_name") + " " + p_max_pu.columns.get_level_values("technology")
p_max_pu = p_max_pu.resample("3h").mean()
p_max_pu

,Mindanao wind-onshore,Mindanao wind-offshore,Mindanao hydro-reservoir-and-run-of-river,Mindanao photovoltaic,Luzon wind-onshore,Luzon wind-offshore,Luzon hydro-reservoir-and-run-of-river,Luzon photovoltaic,Visayas wind-onshore,Visayas wind-offshore,Visayas hydro-reservoir-and-run-of-river,Visayas photovoltaic
2025-01-01 06:00:00,0.840000,1.000000,0.4032,0.200000,0.800000,1.000000,0.2142,0.400000,0.960000,1.000000,0.2700,0.300000
2025-01-01 09:00:00,0.856667,1.000000,0.4032,0.366667,0.843333,1.000000,0.2142,0.566667,0.970000,1.000000,0.2700,0.566667
2025-01-01 12:00:00,0.813333,1.000000,0.4032,0.266667,0.866667,1.000000,0.2142,0.566667,0.960000,1.000000,0.2700,0.400000
2025-01-01 15:00:00,0.683333,1.000000,0.4032,0.033333,0.806667,1.000000,0.2142,0.133333,0.903333,1.000000,0.2700,0.100000
2025-01-01 18:00:00,0.673333,1.000000,0.4032,0.000000,0.710000,1.000000,0.2142,0.000000,0.886667,1.000000,0.2700,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-31 18:00:00,0.280000,0.562667,0.3654,0.000000,0.316667,0.634667,0.4536,0.000000,0.193333,0.384000,0.4266,0.000000
2025-12-31 21:00:00,0.333333,0.672667,0.3654,0.000000,0.310000,0.622667,0.4536,0.000000,0.220000,0.436667,0.4266,0.000000
2026-01-01 00:00:00,0.280000,0.556667,0.3654,0.000000,0.286667,0.578000,0.4536,0.000000,0.280000,0.561333,0.4266,0.000000
2026-01-01 03:00:00,0.210000,0.420000,0.3654,0.000000,0.243333,0.485333,0.4536,0.000000,0.300000,0.595333,0.4266,0.000000


In [22]:
generation = pd.read_csv("../data/Installed Capacity.csv").query("year == 2025 and interconnector.isna()").dropna(how='all', axis=1)
generation.set_index(["node_name", "technology"], inplace=True)
generation.rename(columns={"default_data": "p_nom"}, inplace=True)
generation = generation[["p_nom"]]

In [23]:
p_nom_max = pd.read_csv("../data/Renewable Potential.csv").set_index(["node_name", "technology"])["default_data"]
generation["p_nom_max"] = p_nom_max

In [24]:
efficiency = pd.read_csv("../data/Fuel Use Rate.csv").query("year == 2025").set_index(["node_name", "technology"])["default_data"].dropna()**-1
generation["efficiency"] = efficiency

In [25]:
lifetime = pd.read_csv("../data/Operating Life.csv").query("year == 2025 and technology != 'transmission'").set_index(["node_name", "technology"])["default_data"].dropna()
generation["lifetime"] = lifetime

In [26]:
vom = pd.read_csv("../data/Variable Operating Cost.csv").query("year == 2025 and technology != 'transmission'").set_index(["node_name", "technology"])["default_data"].dropna()
generation["vom"] = vom

In [27]:
overnight_cost = pd.read_csv("../data/Generator Capital Cost.csv").query("year == 2025 and technology != 'transmission'").set_index(["node_name", "technology"])["default_data"].dropna()
generation["overnight_cost"] = overnight_cost

In [28]:
ramp_limit_up = pd.read_csv("../data/Ramp Up Rate.csv").set_index(["node_name", "technology"])["default_data"].dropna()
generation["ramp_limit_up"] = ramp_limit_up

ramp_limit_down = pd.read_csv("../data/Ramp Down Rate.csv").set_index(["node_name", "technology"])["default_data"].dropna()
generation["ramp_limit_down"] = ramp_limit_down

In [29]:
co2_emissions = pd.read_csv("../data/Emissions Rate.csv").query("year == 2025").set_index(["node_name", "technology"])["default_data"].dropna()
generation["co2_emissions"] = co2_emissions * generation["efficiency"]


In [30]:

fuel_price = pd.read_csv("../data/Fuel Price.csv").query("year == 2025 and node_name == 'Luzon'").set_index("commodity")["default_data"].dropna()
TECH_COMMODITY_MAP = {
    "coal-ultrasupercritical": "coal",
    "bioenergy-unspecified": "bioenergy",
    "coal-subcritical": "coal",
    "oil-unspecified": "oil",
    "coal-supercritical": "coal",
    "gas-open-cycle-gas-turbine": "gas",
    "coal-circulating-fluidized-bed": "coal",
    "gas-combined-cycle": "gas",
    "gas-ccs": "gas",
}
generation["fuel_cost"] = generation.index.get_level_values("technology").map(TECH_COMMODITY_MAP).map(fuel_price)


In [31]:
generation = generation.query("technology not in ['import', 'export']").reset_index()
generation.index = generation.node_name + " " + generation.technology

generation["p_nom_max"] = generation["p_nom_max"].fillna(np.inf)
generation.loc[generation.technology == "nuclear", ["efficiency", "fuel_cost"]] = 0.33, 7.5
generation["efficiency"] = generation["efficiency"].fillna(1)
generation["co2_emissions"] = generation["co2_emissions"].fillna(0)
generation["fuel_cost"] = generation["fuel_cost"].fillna(0)


storage = generation.query("technology in ['battery-unspecified', 'hydro-pumped-storage']").copy()
generation = generation.query("technology not in ['battery-unspecified', 'hydro-pumped-storage']")

In [32]:
n = pypsa.Network()

n.snapshots = p_set.index
n.snapshot_weightings.loc[:, :] = 3

n.add("Bus", "Luzon", x=121.22, y=16.58)
n.add("Bus", "Visayas", x=122.53, y=11.19)
n.add("Bus", "Mindanao", x=124.62, y=7.75)

n.add(
    "Link",
    transmission.index,
    bus0=transmission.bus0,
    bus1=transmission.bus1,
    p_nom=transmission.default_data,
    p_min_pu=-1,
    carrier="transmission",
)

n.add(
    "Load",
    p_set.columns,
    bus=p_set.columns,
    p_set=p_set,
    carrier="AC"
)

n.add(
    "Generator",
    generation.index,
    bus=generation.node_name,
    carrier=generation.technology,
    p_nom=generation.p_nom,
    # p_nom_min=generation.p_nom,
    p_nom_max=generation.p_nom_max,
    efficiency=generation.efficiency,
    lifetime=generation.lifetime,
    capital_cost=annuity(0.1, generation.lifetime) * generation.overnight_cost,
    marginal_cost=generation.fuel_cost / generation.efficiency + generation.vom,
    # ramp_limit_up=generation.ramp_limit_up,
    # ramp_limit_down=generation.ramp_limit_down,
    p_max_pu=p_max_pu.reindex(columns=generation.index).fillna(1)
)

n.add(
    "StorageUnit",
    storage.index,
    bus=storage.node_name,
    p_nom=storage.p_nom,
    # p_nom_min=storage.p_nom,
    p_nom_max=storage.p_nom_max.fillna(np.inf),
    max_hours=storage.technology.map({"battery-unspecified": 3, "hydro-pumped-storage": 10}),
    efficiency_store=storage.efficiency**0.5,
    efficiency_dispatch=storage.efficiency**0.5,
    lifetime=storage.lifetime,
    capital_cost=annuity(0.1, storage.lifetime) * storage.overnight_cost,
    marginal_cost=storage.vom,
    carrier=storage.technology,
    cyclic_state_of_charge=True,
)

# capital_costs = (
#     annuity(0.1, 25) * 1_500_000
#     + annuity(0.1, 25) * 550_000
#     + 168 * annuity(0.1, 40) * 40_000
# )

# n.add(
#     "StorageUnit",
#     p_set.columns + " hydrogen-storage",
#     bus=p_set.columns,
#     carrier="hydrogen-storage",
#     max_hours=168,
#     capital_cost=capital_costs,
#     efficiency_store=0.65,
#     efficiency_dispatch=0.45,
#     cyclic_state_of_charge=True,
# );

n.add(
    "Generator",
    p_set.columns + " green-fuel-turbine",
    bus=p_set.columns,
    carrier="green-fuel-turbine",
    efficiency=0.4,
    lifetime=25,
    capital_cost=annuity(0.1, 25) * 600_000,
    marginal_cost=200,
    e_sum_max=5e6, # 5 TWh
)

In [33]:
TECH_COLORS = {
    "AC": "navy",
    'wind-offshore': "lightskyblue",
    'wind-onshore': "royalblue",
    'photovoltaic': "gold",
    'hydro-reservoir-and-run-of-river': "mediumturquoise",
    'bioenergy-unspecified': "forestgreen",
    'waste': "olive",
    'nuclear': "hotpink",
    'geothermal-unspecified': "sienna",
    'coal-ultrasupercritical': "black",
    'coal-supercritical': "darkgrey",
    'coal-subcritical': "grey",
    'coal-circulating-fluidized-bed': "lightgrey",
    'oil-unspecified': "mediumorchid",
    'gas-combined-cycle': "firebrick",
    'gas-open-cycle-gas-turbine': "lightcoral",
    'gas-ccs': "coral",
    'battery-unspecified': "palegreen",
    'hydro-pumped-storage': "teal",
}

carriers = pd.concat([
    generation.groupby("technology").co2_emissions.median(),
    storage.groupby("technology").co2_emissions.median(),
])

n.add(
    "Carrier",
    carriers.index,
    co2_emissions=carriers,
    color=carriers.index.map(TECH_COLORS),
)

n.add("Carrier", "AC", color="navy")
n.add("Carrier", "transmission", color="slateblue")
# n.add("Carrier", "hydrogen-storage", color="purple")
n.add("Carrier", "green-fuel-turbine", color="mediumpurple")

n.export_to_excel("phl-network.nc")


INFO:pypsa.network.io:Exported network 'Unnamed Network' saved to 'phl-network.nc contains: loads, storage_units, links, buses, generators, carriers
